In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append('../')
print(sys.path[-3:])

['', '/home/hieutt/miniconda3/envs/torchtf/lib/python3.9/site-packages', '../']


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report


def evaluate_from_cm(cm, target_names, model_name="Model", digits=4, print_result=True):
    """
    Evaluate classification performance directly from a confusion matrix.

    Args:
        cm: np.ndarray, shape [num_classes, num_classes]
            Confusion matrix where rows are true labels and columns are predicted labels.
        target_names: list[str]
            Class names in the same order as the confusion matrix.
        model_name: str
            Model name used for printing.
        digits: int
            Number of decimal digits in sklearn classification_report.
        print_result: bool
            Whether to print classification report and FNR.

    Returns:
        result: dict
            Contains report_text, report_dict, fnr_dict, fnr_df, accuracy,
            macro_f1, weighted_f1.
    """

    cm = np.asarray(cm)
    num_classes = cm.shape[0]

    if cm.shape[0] != cm.shape[1]:
        raise ValueError("Confusion matrix must be square.")

    if len(target_names) != num_classes:
        raise ValueError("Length of target_names must match number of classes in cm.")

    # Instead of expanding y_true/y_pred, use sample_weight.
    true_idx, pred_idx = np.indices(cm.shape)
    y_true = true_idx.ravel()
    y_pred = pred_idx.ravel()
    weights = cm.ravel()

    report_text = classification_report(
        y_true,
        y_pred,
        sample_weight=weights,
        target_names=target_names,
        digits=digits,
        zero_division=0
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        sample_weight=weights,
        target_names=target_names,
        digits=digits,
        output_dict=True,
        zero_division=0
    )

    # False Negative Rate per class
    fnr_dict = {}
    fnr_rows = []

    for i, name in enumerate(target_names):
        TP = cm[i, i]
        FN = np.sum(cm[i, :]) - TP

        if TP + FN > 0:
            fnr = FN / (TP + FN)
        else:
            fnr = 0.0

        fnr_dict[name] = fnr

        fnr_rows.append({
            "class": name,
            "TP": int(TP),
            "FN": int(FN),
            "FNR": fnr,
            "FNR (%)": fnr * 100
        })

    fnr_df = pd.DataFrame(fnr_rows)

    result = {
        "model_name": model_name,
        "report_text": report_text,
        "report_dict": report_dict,
        "fnr_dict": fnr_dict,
        "fnr_df": fnr_df,
        "accuracy": report_dict["accuracy"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
        "macro_precision": report_dict["macro avg"]["precision"],
        "macro_recall": report_dict["macro avg"]["recall"],
        "weighted_precision": report_dict["weighted avg"]["precision"],
        "weighted_recall": report_dict["weighted avg"]["recall"],
    }

    if print_result:
        print(f"\nClassification Report: {model_name}")
        print(report_text)

        print("False Negative Rate (FNR) per class:")
        for _, row in fnr_df.iterrows():
            print(f"{row['class']:12s}: {row['FNR (%)']:.2f}%")

    return result

In [3]:
target_names = [
    "Normal",
    "Combined",
    "DoS",
    "Fuzzy",
    "Gear",
    "Interval",
    "RPM",
    "Speed",
    "Standstill",
    "Systematic"
]

## Confusion matrix

In [4]:
baseline_cms = {}

In [19]:
baseline_cms["GCN"] = np.array([
    [13656,    72,     0,     2,   151,     1,    13,    61,     0,     1],
    [   18,  4439,     0,     0,     0,     0,     0,     0,     0,     0],
    [    1,     0,   666,     0,     0,     0,     0,     0,     0,     0],
    [    2,     0,     0,   862,     0,     0,     0,     0,     0,     0],
    [    3,     0,     0,     0,   701,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,     0,  6362,     0,     1,     0,     0],
    [  213,     0,     0,     0,     0,     0,  1108,     7,     0,     0],
    [    8,     0,     0,     0,     0,     0,     0,  1586,     0,     0],
    [    4,     0,     0,     0,     0,     0,     0,     0,   613,     0],
    [   19,     0,     0,    11,     0,     0,     0,     0,     0,   957]
])

baseline_cms["GraphSAGE"] = np.array([
    [13688,   157,     0,     2,    29,    11,    34,    35,     0,     1],
    [   13,  4444,     0,     0,     0,     0,     0,     0,     0,     0],
    [    1,     0,   666,     0,     0,     0,     0,     0,     0,     0],
    [    3,     0,     0,   861,     0,     0,     0,     0,     0,     0],
    [    8,     0,     0,     0,   696,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,     0,  6363,     0,     0,     0,     0],
    [   86,     0,     0,     0,     0,     0,  1241,     1,     0,     0],
    [   14,     9,     0,     0,     0,     0,     0,  1571,     0,     0],
    [    4,     0,     0,     0,     0,     0,     0,     0,   613,     0],
    [   15,     0,     0,     4,     0,     0,     0,     0,     0,   968]
])

baseline_cms["R-GCN"] = np.array([
    [13807,    33,     0,     2,    34,     5,    60,    15,     0,     1],
    [   25,  4432,     0,     0,     0,     0,     0,     0,     0,     0],
    [    0,     0,   666,     0,     0,     1,     0,     0,     0,     0],
    [    2,     0,     0,   862,     0,     0,     0,     0,     0,     0],
    [   12,     0,     0,     0,   691,     1,     0,     0,     0,     0],
    [    2,     0,     0,     0,     0,  6361,     0,     0,     0,     0],
    [   64,     0,     0,     0,     0,     0,  1263,     1,     0,     0],
    [   15,     1,     0,     0,     0,     0,     0,  1578,     0,     0],
    [    7,     0,     0,     0,     0,     0,     0,     0,   610,     0],
    [   15,     0,     0,     1,     0,     0,     0,     0,     0,   971]
])

baseline_cms["GIN"] = np.array([
    [13828,    26,     0,     1,    41,     0,    41,    18,     0,     2],
    [   21,  4436,     0,     0,     0,     0,     0,     0,     0,     0],
    [    1,     0,   666,     0,     0,     0,     0,     0,     0,     0],
    [    1,     0,     0,   859,     0,     0,     0,     0,     0,     4],
    [    3,     0,     0,     0,   701,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,     0,  6363,     0,     0,     0,     0],
    [   50,     0,     0,     0,     0,     0,  1278,     0,     0,     0],
    [   12,     0,     0,     0,     0,     0,     1,  1581,     0,     0],
    [    4,     0,     0,     0,     0,     0,     0,     0,   613,     0],
    [    8,     0,     0,     0,     0,     0,     0,     0,     0,   979]
])

# R-GCN - 16
baseline_cms["GATv2"] = np.array([
    [13853,    34,     0,     2,    40,     2,    13,    10,     2,     1],
    [   22,  4435,     0,     0,     0,     0,     0,     0,     0,     0],
    [    0,     0,   666,     0,     0,     1,     0,     0,     0,     0],
    [    2,     0,     0,   862,     0,     0,     0,     0,     0,     0],
    [    2,     0,     0,     0,   702,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,     0,  6363,     0,     0,     0,     0],
    [   56,     0,     0,     0,     0,     0,  1272,     0,     0,     0],
    [   19,     0,     0,     0,     0,     0,     0,  1575,     0,     0],
    [    2,     0,     0,     0,     0,     0,     0,     0,   615,     0],
    [   14,     0,     0,     0,     0,     0,     0,     0,     0,   973]
])

# GraphSAGE - 33
baseline_cms["Edge-GATv2"] = np.array([
    [13840,    40,     0,     1,    25,     2,     9,    31,     0,     9],
    [   11,  4446,     0,     0,     0,     0,     0,     0,     0,     0],
    [    0,     0,   667,     0,     0,     0,     0,     0,     0,     0],
    [    0,     0,    0,   864,     0,     0,     0,     0,     0,     0],
    [    0,     0,    0,     0,   704,     0,     0,     0,     0,     0],
    [    0,     0,    0,     0,     0,  6363,     0,     0,     0,     0],
    [    5,     0,    0,     0,     0,     0,  1323,     0,     0,     0],
    [    7,     0,    0,    0,     0,     0,     0,  1587,     0,     0],
    [    1,     0,    0,     0,     0,     0,     0,     0,   616,     0],
    [    2,     0,    0,     0,     0,     0,     0,     0,     0,   985]
])

# GraphSAGE - 38
baseline_cms["Graph-Transformer"] = np.array([
    [13899,    28,     0,     1,     1,     5,     1,    20,     0,     2],
    [   14,  4443,     0,     0,     0,     0,     0,     0,     0,     0],
    [    0,     0,   667,     0,     0,     0,     0,     0,     0,     0],
    [    1,     0,     0,   863,     0,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,   704,     0,     0,     0,     0,     0],
    [    0,     0,     0,     0,     0,  6363,     0,     0,     0,     0],
    [    3,     0,     0,     0,     0,     0,  1325,     0,     0,     0],
    [    5,     0,     0,     0,     0,     0,     0,   1589,     0,     0],
    [    0,     0,     0,     0,     0,     0,     0,     0,   617,     0],
    [    6,     0,     0,     0,     0,     0,     0,     0,     0,   981]
])

## GCN

In [11]:
gcn_cm = baseline_cms["GCN"]
result = evaluate_from_cm(
    cm=gcn_cm,
    target_names=target_names,
    model_name="GCN",
    digits=4,
    print_result=True
)


Classification Report: GCN
              precision    recall  f1-score   support

      Normal     0.9808    0.9784    0.9796   13957.0
    Combined     0.9840    0.9960    0.9900    4457.0
         DoS     1.0000    0.9985    0.9992     667.0
       Fuzzy     0.9851    0.9977    0.9914     864.0
        Gear     0.8228    0.9957    0.9010     704.0
    Interval     0.9998    0.9998    0.9998    6363.0
         RPM     0.9884    0.8343    0.9049    1328.0
       Speed     0.9583    0.9950    0.9763    1594.0
  Standstill     1.0000    0.9935    0.9967     617.0
  Systematic     0.9990    0.9696    0.9841     987.0

    accuracy                         0.9814   31538.0
   macro avg     0.9718    0.9759    0.9723   31538.0
weighted avg     0.9822    0.9814    0.9813   31538.0

False Negative Rate (FNR) per class:
Normal      : 2.16%
Combined    : 0.40%
DoS         : 0.15%
Fuzzy       : 0.23%
Gear        : 0.43%
Interval    : 0.02%
RPM         : 16.57%
Speed       : 0.50%
Standstill  : 0

## GraphSAGE

In [12]:
graph_sage_cm = baseline_cms["GraphSAGE"]
result = evaluate_from_cm(
    cm=graph_sage_cm,
    target_names=target_names,
    model_name="GraphSAGE",
    digits=4,
    print_result=True
)


Classification Report: GraphSAGE
              precision    recall  f1-score   support

      Normal     0.9896    0.9807    0.9851   13957.0
    Combined     0.9640    0.9971    0.9803    4457.0
         DoS     1.0000    0.9985    0.9992     667.0
       Fuzzy     0.9931    0.9965    0.9948     864.0
        Gear     0.9600    0.9886    0.9741     704.0
    Interval     0.9983    1.0000    0.9991    6363.0
         RPM     0.9733    0.9345    0.9535    1328.0
       Speed     0.9776    0.9856    0.9816    1594.0
  Standstill     1.0000    0.9935    0.9967     617.0
  Systematic     0.9990    0.9807    0.9898     987.0

    accuracy                         0.9865   31538.0
   macro avg     0.9855    0.9856    0.9854   31538.0
weighted avg     0.9866    0.9865    0.9864   31538.0

False Negative Rate (FNR) per class:
Normal      : 1.93%
Combined    : 0.29%
DoS         : 0.15%
Fuzzy       : 0.35%
Gear        : 1.14%
Interval    : 0.00%
RPM         : 6.55%
Speed       : 1.44%
Standstill

## R-GCN

In [14]:
r_gcn_cm = baseline_cms["R-GCN"]
result = evaluate_from_cm(
    cm=r_gcn_cm,
    target_names=target_names,
    model_name="R-GCN",
    digits=4,
    print_result=True
)


Classification Report: R-GCN
              precision    recall  f1-score   support

      Normal     0.9898    0.9893    0.9895   13957.0
    Combined     0.9924    0.9944    0.9934    4457.0
         DoS     1.0000    0.9985    0.9992     667.0
       Fuzzy     0.9965    0.9977    0.9971     864.0
        Gear     0.9531    0.9815    0.9671     704.0
    Interval     0.9989    0.9997    0.9993    6363.0
         RPM     0.9546    0.9511    0.9528    1328.0
       Speed     0.9900    0.9900    0.9900    1594.0
  Standstill     1.0000    0.9887    0.9943     617.0
  Systematic     0.9990    0.9838    0.9913     987.0

    accuracy                         0.9906   31538.0
   macro avg     0.9874    0.9875    0.9874   31538.0
weighted avg     0.9906    0.9906    0.9906   31538.0

False Negative Rate (FNR) per class:
Normal      : 1.07%
Combined    : 0.56%
DoS         : 0.15%
Fuzzy       : 0.23%
Gear        : 1.85%
Interval    : 0.03%
RPM         : 4.89%
Speed       : 1.00%
Standstill  : 

## GIN

In [15]:
gin_cm = baseline_cms["GIN"]
result = evaluate_from_cm(
    cm=gin_cm,
    target_names=target_names,
    model_name="GIN",
    digits=4,
    print_result=True
)


Classification Report: GIN
              precision    recall  f1-score   support

      Normal     0.9928    0.9908    0.9918   13957.0
    Combined     0.9942    0.9953    0.9947    4457.0
         DoS     1.0000    0.9985    0.9992     667.0
       Fuzzy     0.9988    0.9942    0.9965     864.0
        Gear     0.9447    0.9957    0.9696     704.0
    Interval     1.0000    1.0000    1.0000    6363.0
         RPM     0.9682    0.9623    0.9653    1328.0
       Speed     0.9887    0.9918    0.9903    1594.0
  Standstill     1.0000    0.9935    0.9967     617.0
  Systematic     0.9939    0.9919    0.9929     987.0

    accuracy                         0.9926   31538.0
   macro avg     0.9881    0.9914    0.9897   31538.0
weighted avg     0.9926    0.9926    0.9926   31538.0

False Negative Rate (FNR) per class:
Normal      : 0.92%
Combined    : 0.47%
DoS         : 0.15%
Fuzzy       : 0.58%
Gear        : 0.43%
Interval    : 0.00%
RPM         : 3.77%
Speed       : 0.82%
Standstill  : 0.

In [18]:
gat_v2_cm = baseline_cms["GATv2"]
result = evaluate_from_cm(
    cm=gat_v2_cm,
    target_names=target_names,
    model_name="GATv2",
    digits=4,
    print_result=True
)


Classification Report: GATv2
              precision    recall  f1-score   support

      Normal     0.9916    0.9925    0.9921   13957.0
    Combined     0.9924    0.9951    0.9937    4457.0
         DoS     1.0000    0.9985    0.9992     667.0
       Fuzzy     0.9977    0.9977    0.9977     864.0
        Gear     0.9461    0.9972    0.9710     704.0
    Interval     0.9995    1.0000    0.9998    6363.0
         RPM     0.9899    0.9578    0.9736    1328.0
       Speed     0.9937    0.9881    0.9909    1594.0
  Standstill     0.9968    0.9968    0.9968     617.0
  Systematic     0.9990    0.9858    0.9924     987.0

    accuracy                         0.9930   31538.0
   macro avg     0.9907    0.9909    0.9907   31538.0
weighted avg     0.9930    0.9930    0.9930   31538.0

False Negative Rate (FNR) per class:
Normal      : 0.75%
Combined    : 0.49%
DoS         : 0.15%
Fuzzy       : 0.23%
Gear        : 0.28%
Interval    : 0.00%
RPM         : 4.22%
Speed       : 1.19%
Standstill  : 

In [20]:
edge_gat_v2_cm = baseline_cms["Edge-GATv2"]
result = evaluate_from_cm(
    cm=edge_gat_v2_cm,
    target_names=target_names,
    model_name="Edge-GATv2",
    digits=4,
    print_result=True
)


Classification Report: Edge-GATv2
              precision    recall  f1-score   support

      Normal     0.9981    0.9916    0.9949   13957.0
    Combined     0.9911    0.9975    0.9943    4457.0
         DoS     1.0000    1.0000    1.0000     667.0
       Fuzzy     0.9988    1.0000    0.9994     864.0
        Gear     0.9657    1.0000    0.9826     704.0
    Interval     0.9997    1.0000    0.9998    6363.0
         RPM     0.9932    0.9962    0.9947    1328.0
       Speed     0.9808    0.9956    0.9882    1594.0
  Standstill     1.0000    0.9984    0.9992     617.0
  Systematic     0.9909    0.9980    0.9944     987.0

    accuracy                         0.9955   31538.0
   macro avg     0.9918    0.9977    0.9948   31538.0
weighted avg     0.9955    0.9955    0.9955   31538.0

False Negative Rate (FNR) per class:
Normal      : 0.84%
Combined    : 0.25%
DoS         : 0.00%
Fuzzy       : 0.00%
Gear        : 0.00%
Interval    : 0.00%
RPM         : 0.38%
Speed       : 0.44%
Standstil

In [21]:
graph_transformer_cm = baseline_cms["Graph-Transformer"]
result = evaluate_from_cm(
    cm=graph_transformer_cm,
    target_names=target_names,
    model_name="Graph-Transformer",
    digits=4,
    print_result=True
)


Classification Report: Graph-Transformer
              precision    recall  f1-score   support

      Normal     0.9979    0.9958    0.9969   13957.0
    Combined     0.9937    0.9969    0.9953    4457.0
         DoS     1.0000    1.0000    1.0000     667.0
       Fuzzy     0.9988    0.9988    0.9988     864.0
        Gear     0.9986    1.0000    0.9993     704.0
    Interval     0.9992    1.0000    0.9996    6363.0
         RPM     0.9992    0.9977    0.9985    1328.0
       Speed     0.9876    0.9969    0.9922    1594.0
  Standstill     1.0000    1.0000    1.0000     617.0
  Systematic     0.9980    0.9939    0.9959     987.0

    accuracy                         0.9972   31538.0
   macro avg     0.9973    0.9980    0.9977   31538.0
weighted avg     0.9972    0.9972    0.9972   31538.0

False Negative Rate (FNR) per class:
Normal      : 0.42%
Combined    : 0.31%
DoS         : 0.00%
Fuzzy       : 0.12%
Gear        : 0.00%
Interval    : 0.00%
RPM         : 0.23%
Speed       : 0.31%
St

In [22]:
all_results = {}

for model_name, cm in baseline_cms.items():
    cm = np.asarray(cm)

    if cm.ndim != 2 or cm.shape[0] != cm.shape[1]:
        print(f"Skip {model_name}: invalid CM shape {cm.shape}")
        continue

    result = evaluate_from_cm(
        cm=cm,
        target_names=target_names,
        model_name=model_name,
        digits=4,
        print_result=True
    )
    all_results[model_name] = result


Classification Report: GCN
              precision    recall  f1-score   support

      Normal     0.9808    0.9784    0.9796   13957.0
    Combined     0.9840    0.9960    0.9900    4457.0
         DoS     1.0000    0.9985    0.9992     667.0
       Fuzzy     0.9851    0.9977    0.9914     864.0
        Gear     0.8228    0.9957    0.9010     704.0
    Interval     0.9998    0.9998    0.9998    6363.0
         RPM     0.9884    0.8343    0.9049    1328.0
       Speed     0.9583    0.9950    0.9763    1594.0
  Standstill     1.0000    0.9935    0.9967     617.0
  Systematic     0.9990    0.9696    0.9841     987.0

    accuracy                         0.9814   31538.0
   macro avg     0.9718    0.9759    0.9723   31538.0
weighted avg     0.9822    0.9814    0.9813   31538.0

False Negative Rate (FNR) per class:
Normal      : 2.16%
Combined    : 0.40%
DoS         : 0.15%
Fuzzy       : 0.23%
Gear        : 0.43%
Interval    : 0.02%
RPM         : 16.57%
Speed       : 0.50%
Standstill  : 0

In [23]:
import os
import numpy as np
import pandas as pd


def build_classwise_result_table(
    all_results,
    target_names,
    model_order=None,
    save_path="classwise_baseline_results.csv",
    digits=4
):
    """
    Build class-wise result table from all_results returned by evaluate_from_cm().

    Output format:
    Attack Type | Model | FNR (%) | Rec | Prec | F1 | Support
    """

    rows = []

    if model_order is None:
        model_order = list(all_results.keys())
    else:
        # Keep only models that already exist in all_results
        model_order = [m for m in model_order if m in all_results]

    for class_name in target_names:
        for model_name in model_order:
            result = all_results[model_name]
            report_dict = result["report_dict"]
            fnr_dict = result["fnr_dict"]

            if class_name not in report_dict:
                print(f"Skip {model_name} - {class_name}: class not found in report_dict")
                continue

            precision = report_dict[class_name]["precision"]
            recall = report_dict[class_name]["recall"]
            f1 = report_dict[class_name]["f1-score"]
            support = report_dict[class_name]["support"]
            fnr = fnr_dict[class_name] * 100.0

            rows.append({
                "Attack Type": class_name,
                "Model": model_name,
                "FNR (%)": round(fnr, 2),
                "Rec": round(recall, digits),
                "Prec": round(precision, digits),
                "F1": round(f1, digits),
                "Support": int(support),
            })

    df = pd.DataFrame(rows)

    # Save CSV
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    df.to_csv(save_path, index=False)

    print(f"Saved class-wise result table to: {save_path}")
    return df

In [24]:
model_order = [
    "GCN",
    "GraphSAGE",
    "R-GCN",
    "GIN",
    "GATv2",
    "Edge-GATv2",
    "Graph-Transformer",
    "MRGAT-KAN (Proposed)"
]

classwise_df = build_classwise_result_table(
    all_results=all_results,
    target_names=target_names,
    model_order=model_order,
    save_path="classwise_baseline_results.csv",
    digits=4
)

classwise_df

Saved class-wise result table to: classwise_baseline_results.csv


,Attack Type,Model,FNR (%),Rec,Prec,F1,Support
0,Normal,GCN,2.16,0.9784,0.9808,0.9796,13957
1,Normal,GraphSAGE,1.93,0.9807,0.9896,0.9851,13957
2,Normal,R-GCN,1.07,0.9893,0.9898,0.9895,13957
3,Normal,GIN,0.92,0.9908,0.9928,0.9918,13957
4,Normal,GATv2,0.75,0.9925,0.9916,0.9921,13957
...,...,...,...,...,...,...,...
65,Systematic,R-GCN,1.62,0.9838,0.9990,0.9913,987
66,Systematic,GIN,0.81,0.9919,0.9939,0.9929,987
67,Systematic,GATv2,1.42,0.9858,0.9990,0.9924,987
68,Systematic,Edge-GATv2,0.20,0.9980,0.9909,0.9944,987
